# DABN13 - Assignment 2

This assignment consists of two parts. Part 1 covers principal component regression (PCR), while Part 2 covers regularization methods. The two parts are independent of each other and are based on different data sets.

---------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------

# <span style="color:blue"> Part 1: PCR</span>

We continue with the problem of forecasting market return that we illustrated in Lecture 5. In order to train the implementation of PCR, we will  replicate our previous results with a slight twist. More specifically, we will choose tuning parameters via cross-validation. Additionally, the training sets used for model evaluation are defined differently. While the example in Lecture 5 defined an expanding window of training sets with fixed starting period, we are going to use a rolling window that moves both the start and end date of the training set.

The data for this lab is provided by two csv files. 
The file *sorted_portfolios100.csv* contains the monthly returns of 100 equally weighted portfolios sorted by size and the book-to-market ratio. The data is taken from Kenneth French's data library and missing values have been imputed. The time period covered is January 1960 until December 2009
The file *twelve_month_returns.csv* contains 12-month returns on a value-weighted market portfolio. This series takes moving 12-month sums of returns of the U.S. market factor, as provided on Kenneth French's data library. The entry in row $t$ of the dataset corresponds to the market returns over the months $t+1$ until $t+12$. Accordingly, the first observed value in our sample is the 12-month return over the period February 1960 - January 1961. The last observation covers the period January-December 2010.

To begin with, we import both your outcome as well as the 100 predictors into Python using the code below. You might be required to modify the file path.


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from os import chdir

#chdir('?') Adjust directory here
os.chdir('c:\\Users\\Vicente\\Desktop\\Lund - MSc Economics\\ML - From a Regression perspective\\ml-course\\data') # Change working directory here
mkt_ret_12m = pd.read_csv("twelve_month_returns.csv").iloc[:,1:]  # this is our y-variable
portfolios  = pd.read_csv("sorted_portfolios100.csv").iloc[:,1:]  # these are our x-variables


mkt_ret_12m_train = mkt_ret_12m.iloc[:540, :]
portfolios_train  = portfolios.iloc[:540, :]  

mkt_ret_12m_test = mkt_ret_12m.iloc[540:, :]
portfolios_test  = portfolios.iloc[540:, :]

ModuleNotFoundError: No module named 'seaborn'

## 1a) Correlation matrix

**TASKS**

1. Compute a Pearson correlation matrix for all variables in the `portfolios_train` dataset using the `corr()` method (see the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)). The code for generating the heatmap from the correlation matrix has already been provided, do not modify it.
3. Write a short interpretation of the correlation heatmap and store it in the string variable `your_interpretation_of_correlation_matrix`. Focus on the overall correlation pattern: Are many or few variables correlated with each other? Are the correlations generally strong or weak?

In [ ]:
# 1.
corr = ??

# do not change the next 11 lines
plt.figure(figsize = (7, 5))
sns.heatmap(
    corr,
    cmap = "coolwarm",
    center = 0,
    annot = False,
    square = True
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

# 2. 
your_interpretation_of_correlation_matrix = """ ?  """

## 1b) Setting up a PCR pipeline in `sklearn`

Principal component regression is not available in scikit-learn as a distinctive learning algorithm. This is no problem because it is the combination of three simple components:

1. standardizing inputs,
2. obtaining $k$ principal components from the inputs,
3. running a linear regression of the output on the $k$ principal components (and a constant).

All three operations above are implemented in sklearn. In order to put them together, we will use the very convenient `Pipeline()` functionality (see the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)).
Pipelines allow us to specify a custom sequence of actions. An existing pipeline can be fit, similar to the specification of a regression model. If we do that, all steps of the pipeline are executed in their specified order.

We will create and fit a pipeline for PCR. First, we need to specify the three actions mentioned above individually. This can be done by using the `StandardScaler()`, `PCA()` and `LinearRegression()` functions from several `sklearn` modules. I have already written code below to import these functions.

*Hint*: In the ISLP book (2023 Python edition), you can find inspiration in section 6.5.3. We won't do the exact same thing, but it can be of help when learning how to specify pipelines, how to use `PCA()` etc.

**TASKS**
1. Create 3 objects, `standardizer_1b`, `pca_1b` and `lm_1b` which specify input standardization, PCA **without specifying the number of components**, a linear regression model, respectively. 

2. Specify the 3 steps of our pipeline and give them suitable names. Save the resulting object as `pcr_pipe`. I have written code that clones this Pipeline multiple times. Later in Part 1, we will need several pipelines with largely the same structure. Rather than making you repeatedly write the same pipeline specification, I've written code that clones your `pcr_pipe` multiple times.

4. Use `set_params()` on `pcr_pipe_1b` to specify the number of components to be **5**. 

5. Apply the `fit()`-method to `pcr_pipe_1b` to execute the entire pipeline. Do this using `mkt_ret_12m_train`and `portfolios_train` as your output and inputs, respectively. Save the fitted pipeline as `pcr_pipe_fit_1b`. Note that this is now fitted using 5 principal components.



In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# 1.
standardizer_1b = ??
pca_1b = ??
lm_1b = ??

# 2. 
pcr_pipe = Pipeline(steps = [(??), (??), (??)])

# Do not change the 5 lines below (see the note above)
pcr_pipe_1b = clone(pcr_pipe)
pcr_pipe_1c_no_lrm = clone(pcr_pipe[:-1])
pcr_pipe_1c = clone(pcr_pipe)
pcr_pipe_1d = clone(pcr_pipe)
pcr_pipe_1e_no_pca = clone(Pipeline([pcr_pipe.steps[0], pcr_pipe.steps[2]]))

# 3.
pcr_pipe_1b.??(pca__n_components = ??)

# 4.
pcr_pipe_fit_1b  = ??.fit(X = ??, y = ??)  # we will use this pipeline later

## 1c) Cumulative explained variance

In the previous task, the number of principal components we used was an arbitrary choice. A more informed way to choose the number of principal components is to examine the cumulative explained variance, which you will do in this exercise.

In the first code chunk I ask you to not modify things. Here, we standardize the data and fit the PCA algorithm **without** a specific number of components. We do **not** fit a linear regression, because the regression step was removed from the pipeline `pcr_pipe_1c_no_lrm`. Hence, we perform *PCA* and not *PCR*. After the fitting procedure, the code is for plotting the **cumulative explained variance**

**TASKS** 

1. You need to *interpret the plot and suggest an appropriate number of principal components*. Write your answer in the string variable `interpretation_plot_1c`. There is **no single correct answer**; your choice should be motivated based on the features of the plot.

   *HINT*: While it's not in your suggested reading, you can find inspiration on page 514 of the ISLP book (2023 python ed.). The plot shows, for each possible number of principal components, the share of the total variance in the original data that is explained by those components *together*. Use the plot to assess how many components are needed to capture a sufficiently large share of the variance.

2. For `pcr_pipe_1c`, specify `n_components` **as the number you consider reasonable based on the plot**. 

3. Fit `pcr_pipe_1c` to the training data and store as `pcr_pipe_fit_1c`. Note that this pipeline contains the regression step, so we are fitting a linear regression as the last step.

In [ ]:
# Do not change any of the lines below
pcr_pipe_1c_no_lrm.fit(portfolios_train)
pca_1c = pcr_pipe_1c_no_lrm.named_steps['pca']

n_components = np.arange(1, len(pca_1c.explained_variance_ratio_) + 1)
plt.plot(n_components, np.cumsum(pca_1c.explained_variance_ratio_))
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True, linestyle = "--", alpha = 0.7)
plt.xticks(np.arange(0, len(pca_1c.explained_variance_ratio_) + 1, 6))
plt.show()

# 1.
interpretation_plot_1c = """ ?? """

# 2.
pcr_pipe_1c.??(pca__n_components = ??)

# 3
pcr_pipe_fit_1c  = ??.fit(X = ??, y = ??)  # we will use this pipeline later

## *1d) Model tuning*

Another more informed way to choose the number of principal components is to use `sklearn`for model tuning. The tuning parameter of PCR is the number of principal components to use. 

**TASKS**

1. Create a parameter grid for the number of principal components. Use 3, 4, 10, 15, 20, 30, and 35 as candidate values and store the dictionary as `parameter_grid_1d`. (*Hint*: You can check ISLP section 6.5.3 for inspiration on how to define a grid).
   
3. Use `TimeSeriesSplit()` to create the cross-validation splitting strategy `cv_splits_1d`, with these properties:
     - Training folds should always consist of at most 90 data points.
     - The test fold following the training folds should only contain 1 data point.
     - We want a total of 450 splits, so that the training fold window moves through the entire length of `mkt_ret_12m_train` and `portfolios_train`.
     - No gap between training and test folds.
     - Please check the  `TimeSeriesSplit()`-[documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html#sklearn.model_selection.TimeSeriesSplit) to figure out how to set the function inputs.
       
  
4. Use the `GridSearchCV()` function in the `model_selection` module of Scikit-learn to specify the setup for model tuning. To be more specific, we want the following:
     - tune the model pipeline `pcr_pipe_1d`.
     - use the grid of tuning parameter values stored in `parameter_grid_1d`,
     - cross-validation is conducted based on the splits defined in `cv_splits_1d`,
     - performance should be evaluated using the negative root mean square error.
     - the parameter `refit` must be set to its default value ('True'), so do **not** change it.
     - Look up the `GridSearchCV()`-[documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) to figure out how to set the function arguments correctly.
     - Save the resulting setup for model tuning as `pcr_tune_1d`.
       
5. Run the tuning procedure by applying the `fit()`-method to `pcr_tune_1d`. Use `portfolios_train` and `mkt_ret_12m_train` as inputs and output, respectively. `pcr_tune_1d` now contains cross-validation results, the optimal tuning parameter value, the learned model with this value and more.

6. Use the `GridSearchCV()` documentation to find out how to identify the optimal tuning parameter value. In the string variable `optimal_number_components`, report the value and explain which code you used to find it.
   
7. Identify the fitted pipeline corresponding to the optimal tuning parameter value and save it as `pcr_pipe_tuned_1d`. (*Hint*: You need to use one of the Attributes in GridSearchCV)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV

# 1. 
parameter_grid_1d = {??}

# 2.
cv_splits_1d     = TimeSeriesSplit(??)

# 3.
pcr_tune_1d = GridSearchCV(??,
                           ??, 
                           ??, 
                           ??, 
                           n_jobs = -1)

# 4.
??.fit(??) 

# 5.
optimal_number_components = """ ?? """

# 6.
pcr_pipe_tuned_1d = ?? # we will use this pipeline later

## *1e) Test set performance*

So far, we have only used the **training** data set, which consists of the first 540 observations. Once a model has been fitted to the training data, we can use it to generate predictions for new observations using the predict() method.

To evaluate how well our fitted models perform on new data, we compare their predictions with the observed outcomes in the **test** data set. The test data have been kept separate throughout the lab and will now be used as a hold-out sample for evaluating predictive performance. We will evaluate the models using mean squared error (MSE) and R².

I have already fitted a pipeline for a simple linear regression model, called `lm_pipe_fit_1e`. This is a model without any PCA application.

I have also provided a function `evaluate_models`for generating predictions on the test set and evaluating their performance. 

**TASKS**

1. In the a dictionary called `pipelines_1e` you should enter the fitted pipelines from Tasks 1b)–1d). The linear regression pipeline has already been added to the dictionary as an example.
   
3. Create a dictionary containing the two performance metrics, MSE and R². Save it as `metrics_1e`.
   
4. In the string variable `interpretation_function`, write a brief description of what is happening inside the `evaluate_models` function.
   
5. Run the code to get a printed table. In the string variable `comparison_models_1e`, I want you to do the following:
   - briefly interpret the performance metrics
   - compare the PCR models with the standard linear regression model. Do any of the PCR models perform better? If so, why might this be?
   - compare the predictive performance of the different PCR models and discuss why their performance may differ.

     *Hint*: Consider the number of principal components used by the different PCR models and how this number was selected in 1c) versus 1d). What criterion was used in each case, and why might these criteria lead to different test-set performance? Also consider whether your findings from task in 1a) can help explain some results.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Do not change the lines below
lm_pipe_fit_1e = pcr_pipe_1e_no_pca.fit(X = portfolios_train, y = mkt_ret_12m_train)

def evaluate_models(pipelines_dict, metrics_dict, X_test, y_test):
    results = []

    for name, pipeline in pipelines_dict.items():
        y_pred = pipeline.predict(X_test) 

        model_results = {"Model": name}

        for metric_name, metric_function in metrics_dict.items():
            model_results[metric_name] = metric_function(y_test, y_pred)

        results.append(model_results)

    results = (
        pd.DataFrame(results)
          .round(3)
    )

    return results
    
# 1.
pipelines_1e = {
    "Linear Regression": lm_pipe_fit_1e,
    ??,
    ??,
    ??
}

# 2.
metrics_1e = {
    ??
}


results_1e = evaluate_models(
    pipelines_dict = pipelines_1e,
    metrics_dict = metrics_1e,
    X_test = portfolios_test,
    y_test = mkt_ret_12m_test
)

print(results_1e)

# 3. 
interpretation_function = """ ??? """

# 4.
comparison_models_1e = """ ??? """

---------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------

# <span style="color:blue"> Part 2: Regularization Methods</span>

Air carriers regularly update their destination network and a necessary condition for adding a new flight connection to the existing network is that flights on this route are profitable. In addition to costs, profitability is determined by the price at which competitors could cater to the demand for flight transportation between two locations. A data-driven way of determining this would be on the basis of a supervised learning algorithm that has been trained on the fares for existing flight connections and which leverages information about route-specific characteristics. The dataset that we are using in this lab contains such information for more than 600 flight connections between different American cities. More specifically, we have information on the number of required stops, the market concentration on a particular route, Holiday destination status and a number of other characteristics that are detailed on [p.7 in the `mlba` library documentation](https://github.com/gedeck/mlba/blob/main/mlba_2.0.0.pdf). In the following, we will train and tune different supervised learning models that predict the average fare for a flight.


## 2a) Data 

`sklearn` requires us to provide output and inputs separately as data frames containing numerical variables that can be directly put into a cost minimization problem. Provide that by conducting the following steps:

**TASKS**
1. Save the output variable *FARE* as pandas series `y_2a`.

2. The other variables in `Airfares` are the predictors. Save them as a Pandas dataframe `X_2a`. Then, use the `get_dummies()` function in Pandas to turn categorical variables into category dummies. Ensure that the first category of every categorical variable is dropped.

3. We currently have 13 predictors. To create a high-dimensional problem, we want to expand the set of predictors by adding all pairwise interaction terms. Use `PolynomialFeatures()` to specify the transformer `poly2_spec_2a`. Include interactions between pairs of original inputs, but **do not include squared terms or a constant term**. Use `fit_transform()` to apply the transformer to `X_2a`. This gives us a total of 91 predictor variables.

   
8. Use the  `train_test_split()` function in the model selection module of `sklearn` to split `X_2a` and `y_2a` into training and test data sets. Use 70% of the observations for training.



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

Airfares = pd.read_csv("Airfares.csv.gz") # Adjust file path if necessary
Airfares = Airfares.drop(columns = ["S_CODE", "S_CITY", "E_CODE", "E_CITY"])

# 1.
y_2a = ??['FARE']

# 2. 
X_2a = ??.drop(columns = ??)
X_2a = pd.get_dummies(??)

# 3. 
poly2_spec_2a = PolynomialFeatures(??)
X_2a = ??.??(X_2a)

# 4.
X_train_2a, X_test_2a, y_train_2a, y_test_2a = ??


## 2b) Ridge and Lasso: Coefficient paths

We will first examine how the coefficients estimated by Ridge and Lasso change with the strength of regularization.

`lambda_grid` contains 100 candidate values for the regularization parameter $\lambda$. Do not change this.


**TASKS**

1. Use `StandardScaler()` to standardize the `X_train_2a`. Save as `X_train_2b`.
   
5. In the for-loop, one `Ridge` estimator is fitted for each candidate value of $\lambda$ in `lambda_grid`. Specify the sklearn parameter corresponding to $\lambda$, and the attribute containing the fitted coefficient vector. *Hint*: Check the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), particularly the **Parameters** section and the **Attributes** section.

11. Repeat Task 3 using `Lasso`. If you receive a convergence warning, check the `Lasso` [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) and increase the maximum number of iterations until the estimator converges.
   
12. The provided plotting code will use these coefficient vectors to plot the Ridge and Lasso coefficient paths. Run the code. Then interpret the two coefficient paths **separately**. Store your Ridge interpretation in `interpretation_Ridge_path` and your Lasso interpretation in `interpretation_Lasso_path`. For each method, describe how increasing $\lambda$ affects the estimated coefficients and note any characteristic differences in how the coefficients approach zero.

*Do not compare the amount of shrinkage at the same numerical value of $\lambda$ across Ridge and Lasso. The penalty functions differ, so equal numerical values of $\lambda$ do not represent directly comparable amounts of regularization across the two estimators.*


In [ ]:
from sklearn.linear_model import (Ridge, Lasso)

lambda_grid = np.logspace(5, -2, 100)

# 1.
X_train_2b = ??.fit_transform(X_train_2a)

# 2.
coefs_ridge = []
for lambda_candidate in lambda_grid:
    ridge = Ridge(?? = lambda_candidate).fit(X_train_2b, y_train_2a)
    coefs_ridge.append(ridge.??)

# 3. 
coefs_lasso = []
for lambda_candidate in lambda_grid:
    lasso = Lasso(?? = lambda_candidate, max_iter = 1000).fit(X_train_2b, y_train_2a)
    coefs_lasso.append(lasso.??)


# Do not change any of the lines below 
fig, ax = plt.subplots(1, 2, figsize = (10, 4), sharey = True)
for axis, coefs, title in zip(ax,
    [coefs_ridge, coefs_lasso],
    ["Ridge Coefficient Path", "Lasso Coefficient Path"]):
    axis.plot(lambda_grid, coefs, linewidth = 0.9)
    axis.set_xscale("log")
    axis.set_xlim(lambda_grid.min(), lambda_grid.max())
    axis.set_xlabel("Lambda")
    axis.set_title(title)
    axis.grid(True)
ax[0].set_ylabel("Learned coefficients")
plt.tight_layout()
plt.show()


# 4.
interpretation_Ridge_path = """ ?? """
interpretation_Lasso_path = """ ?? """


## 2c) Tuning Ridge, Lasso and Elastic Net in one go!

We have now inspected the coefficient paths for Ridge and Lasso regression. However, we still do not know which value of the regularization parameter ($\lambda$) is optimal for prediction. We therefore want to tune these algorithms using K-fold cross-validation. In addition, we will introduce Elastic Net.

In Part 1 (PCR), we tuned one pipeline specification: (Scaler, PCA, Linear regression). Now we want to tune 3 pipeline specifications: (scaler, Ridge), (scaler, Lasso) and (scaler, Elastic Net). Doing this separately would involve a lot of repetition. Fortunately, GridSearchCV allows us to combine the three specifications into a single parameter grid and tune them together, without having to build separate grids or run the procedure separately for each estimator.


**TASKS**

1. In contrast to Part 1 of this assignment, we no longer have time-ordered data. Therefore, we will not use `TimeSeriesSplit()` to partition the training data. Instead, we will use `KFold()`, which works like the splitting strategy you have seen in the lecture slides. Use  `KFold()` to create a **random** partition into five folds. Set the `random_state` to 5 for the sake of replicability. Save the splitting strategy as `cv_splits_2c`.

   
2. Specify a pipeline that standardizes the data as a first step. Name the second step `model` and use Ridge regression as the estimator. Here, Ridge will only be a placeholder and will be overwritten later. Save the pipeline as `pipe_2c`.
    
3. `param_grid_2c` is a *list* containing three *dictionaries*. Each dictionary should specify one estimator and its candidate hyperparameter values. I have already done this for Ridge. Your task is to complete the dictionaries for Lasso and Elastic Net. For Elastic Net, we want to tune two hyperparameters: the regularization strength ($\lambda$) and the mixing ratio. Therefore, the Elastic Net dictionary must contain 3 items (the first being the model).  *For all 3 estimators*: use the candidate values of $\lambda$ stored in `lambda_grid` (which we specified in task 2b). *For Elastic Net*: use the candidate values of the mixing ratio stored in `elnet_mixing_ratio_grid`.

   
5. Complete the `GridSearchCV()` by setting the splitting strategy to `cv_splits_2c`, and use negative mean squared error as the scoring measure. Do not change the code that has already been provided. Save the resulting GridSearchCV object as `grid_search_2c`.
   
6. Fit `grid_search_2c` to the training data.

7. Extract the cross-validation results from `grid_search_2c` and convert them to a pandas DataFrame. Save the resulting DataFrame as `results_tuning_2c`. The code below this task tidies the DataFrame and adds the model names (don't change it).

8. Look at the Dataframe being printed. In the string variable `best_ridge_lambda_analysis`, do the following:

   - Report the value of $\lambda$ that was selected as optimal by the cross-validation for Ridge regression. 
   - Return to the Ridge coefficient path in Task 2b. Using the optimal value of $\lambda$, describe the extent to which the coefficients are shrunk toward zero relative to a linear regression model without regularization. (The plot is very zoomed out, so you only need to eyeball it, not exact values)

     
9. Look at the DataFrame again. In the string variable `interpretation_CV_results`, explain what the following columns represent: split0_test_score, split1_test_score, split2_test_score, split3_test_score,	split4_test_score,	mean_test_score,	std_test_score. What does each of these values reflect in the cross-validation procedure?



In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNet


# 1.
cv_splits_2c = KFold(??)


# 2.
pipe_2c = Pipeline([
    (??),
    (??)  # placeholder estimator (it will be overwritten later)
])

# 3.
elnet_mixing_ratio_grid = np.linspace(0.0001, 1 , 10)

param_grid_2c = [
    # Ridge dictionary
    {
        'model': [Ridge()],
        'model__alpha': lambda_grid    },
    # Lasso dictionary
    {
        ??,
        ??  },
    # Elastic Net dictionary
    {
        ??,
        ??,
        ??    }
]

# 4.
grid_search_2c = GridSearchCV(estimator = pipe_2c, 
                           param_grid = param_grid_2c, 
                           cv = ??, 
                           scoring = ??, 
                           refit = False,
                           n_jobs = -1)

# 5.
??


# 6.
results_tuning_2c = ??


# Do not change the lines below
results_tuning_2c.insert(
    0, 
    'model', 
    results_tuning_2c['param_model'].apply(lambda x: type(x).__name__))
best_models_2c = results_tuning_2c.loc[
    results_tuning_2c.groupby('model')['mean_test_score'].idxmax()
].reset_index(drop=True)

display(best_models_2c)


# 7. 
best_ridge_lambda_analysis = """ ?? """

# 8. 
interpretation_CV_results = """ ? """

## 2d) Prediction performance evaluation

We have now found the optimal hyperparameter values for Ridge, Lasso, and Elastic Net. Because we tuned the three models together using GridSearchCV with refit=False, no estimator is refitted or stored as the best fitted model. To evaluate the performance of all three models on the untouched test data, we therefore need to fit their pipelines to the training data again using the optimal hyperparameter values. Since this involves repetitive code, I have already provided it below. I have also fitted a simple linear regression without regularization to the training data, which we will use for comparison.

Your tasks are to complete the evaluation of the four fitted pipelines.

**TASKS**
1. We want to evaluate the predictive performance using mean squared error. Specify this in the dictionary `metrics_2d`.
   
3. We will reuse the `evaluate_models` function from the PCA task. I have already specified `pipelines_2d`. Complete the remaining arguments to the function.

   
3. A table showing the MSE on the test set will be printed. Which of the four models performs best? Write your conclusion, including some discussion of the margin between the best-performing model and the remaining models, in the string variable `conclusion_2d`.

   
7. Why do you think one candidate performs better than another? In particular, compare the simple linear regression with the regularized models. Discuss the results in the string variable `discussion_2d`.



In [ ]:
from sklearn.base import clone

# Do not change the code below
pipelines_2d = {}

for _, row in best_models_2c.iterrows():
    pipe = clone(pipe_2c)
    pipe.set_params(**row['params'])
    pipe.fit(X_train_2a, y_train_2a)
    pipelines_2d[row['model']] = pipe

lm_pipe_2d = clone(pipe_2c).set_params(model = LinearRegression())
lm_pipe_2d.fit(X_train_2a, y_train_2a)
pipelines_2d['LinearRegression'] = lm_pipe_2d


# 1. 
metrics_2d = ??

# 2. 
results_2d = evaluate_models(
    pipelines_dict = pipelines_2d,
    metrics_dict = ??,
    X_test = ??,
    y_test = ??
)
print(results_2d)


# 3. 
conclusion_2d = """ ? """

# 4.
discussion_2d = """ ? """

# <span style="color:blue"> AI statement</span>

Please type 'yes' in the string `we_have_used_AI` below if you have used AI in any sense for this assignment. Otherwise, type 'no'. 

A 'yes' or 'no' is enough, you do **not** need to describe which AI tool you used, what you used it for, etc.

In [ ]:
we_have_used_AI = " ?? "